In [1]:
import pandas as pd

df = pd.read_csv("University_Cleaned_mo.csv")

print(df.shape)
df.head()

(4565, 101)


,university_name,QS_Rank,THE_Rank,ARWU_Rank,Enrollment,Undergrad_Students,Postgrad_Students,Student_Faculty_Ratio,Research_Output,International_Students (%),...,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio,country,region,university_type,total_students
0,aalborg university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.0,82.0,49.94,50.25,55.77,16.83,Denmark,Europe,Public,17422.000
1,aalto university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9.0,76.0,38.96,38.59,40.11,9.00,Finland,Europe,Public,16099.000
2,aarhus university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.0,82.0,49.94,50.25,55.77,16.83,Denmark,Europe,Public,23895.000
3,abai kazakh national pedagogical university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.0,236.0,38.08,42.57,37.89,1158.35,Kazakhstan,Asia,Public,13.958
4,abasyn university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import os
print(os.listdir())

['.config', 'University_Cleaned_mo.csv', 'sample_data']


In [3]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv("University_Cleaned_mo.csv")

print(df.shape)
df.head()

(4565, 101)


,university_name,QS_Rank,THE_Rank,ARWU_Rank,Enrollment,Undergrad_Students,Postgrad_Students,Student_Faculty_Ratio,Research_Output,International_Students (%),...,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio,country,region,university_type,total_students
0,aalborg university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.0,82.0,49.94,50.25,55.77,16.83,Denmark,Europe,Public,17422.000
1,aalto university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9.0,76.0,38.96,38.59,40.11,9.00,Finland,Europe,Public,16099.000
2,aarhus university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.0,82.0,49.94,50.25,55.77,16.83,Denmark,Europe,Public,23895.000
3,abai kazakh national pedagogical university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.0,236.0,38.08,42.57,37.89,1158.35,Kazakhstan,Asia,Public,13.958
4,abasyn university,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
required_cols = [
    "qs_rank_2026",
    "the_rank_2026",
    "arwu_rank_2025",
    "citations_count",
    "cited_by_count",
    "h_index",
    "faculty_count",
    "total_students",
    "international_students_count",
    "qs_academic_rep",
    "academic_reputation_score",
    "publications_count",
    "works_count"
]

missing = [c for c in required_cols if c not in df.columns]

print("Missing columns:", missing)

Missing columns: []


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# Convert columns to numeric
# -----------------------------
numeric_cols = [
    "qs_rank_2026","the_rank_2026","arwu_rank_2025",
    "citations_count","cited_by_count","h_index",
    "faculty_count","total_students",
    "international_students_count",
    "qs_academic_rep","academic_reputation_score",
    "publications_count","works_count"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(0)

scaler = MinMaxScaler()

# -----------------------------
# 1. Global Ranking Score
# -----------------------------
rank_score = (
    (1/(df["qs_rank_2026"]+1)) +
    (1/(df["the_rank_2026"]+1)) +
    (1/(df["arwu_rank_2025"]+1))
) / 3

df["Global_Ranking_Score"] = scaler.fit_transform(
    rank_score.values.reshape(-1,1)
)

# -----------------------------
# 2. Research Impact Score
# -----------------------------
research = (
    0.4*df["citations_count"] +
    0.3*df["h_index"] +
    0.3*df["cited_by_count"]
)

df["Research_Impact_Score"] = scaler.fit_transform(
    research.values.reshape(-1,1)
)

# -----------------------------
# 3. Faculty-to-Student Ratio
# -----------------------------
df["Faculty_to_Student_Ratio"] = np.where(
    df["total_students"] > 0,
    df["faculty_count"]/df["total_students"],
    0
)

# -----------------------------
# 4. International Student Percentage
# -----------------------------
df["International_Student_Percentage"] = np.where(
    df["total_students"] > 0,
    (df["international_students_count"]/df["total_students"])*100,
    0
)

# -----------------------------
# 5. Academic Reputation Score
# -----------------------------
academic = (
    df["qs_academic_rep"] +
    df["academic_reputation_score"]
) / 2

df["Academic_Reputation_Score"] = scaler.fit_transform(
    academic.values.reshape(-1,1)
)

# -----------------------------
# 6. Research Productivity Index
# -----------------------------
df["Research_Productivity_Index"] = np.where(
    df["faculty_count"] > 0,
    (df["publications_count"] + df["works_count"]) / df["faculty_count"],
    0
)

print("✅ KPI Generation Completed Successfully!")

print(df[[
    "Global_Ranking_Score",
    "Research_Impact_Score",
    "Faculty_to_Student_Ratio",
    "International_Student_Percentage",
    "Academic_Reputation_Score",
    "Research_Productivity_Index"
]].head())

✅ KPI Generation Completed Successfully!
   Global_Ranking_Score  Research_Impact_Score  Faculty_to_Student_Ratio  \
0                   1.0               0.129650                  0.106762   
1                   1.0               0.046321                  0.086279   
2                   1.0               0.102843                  0.055325   
3                   1.0               0.001885                  3.582175   
4                   1.0               0.013824                  0.000000   

   International_Student_Percentage  Academic_Reputation_Score  \
0                         14.998278                   0.265620   
1                         17.001056                   0.209983   
2                         13.998745                   0.287822   
3                       2385.728614                   0.210425   
4                          0.000000                   0.000000   

   Research_Productivity_Index  
0                     5.638172  
1                     3.675306  
2     

In [6]:
df.to_csv("University_Final_KPI.csv", index=False)

print("✅ University_Final_KPI.csv saved successfully!")

✅ University_Final_KPI.csv saved successfully!
